# Joint Distributions of Independent Variables

Two fair dice make the cleanest possible joint distribution: when $X$ is the first roll and $Y$ the second, the variables are **independent** and the joint PMF *factors* — $P_{XY}(x,y) = P_X(x)\,P_Y(y)$, a table of $\tfrac{1}{36}$ everywhere. But change $Y$ to the **sum** of the dice and the picture transforms: the joint becomes a diagonal band of $\tfrac{1}{36}$-cells full of structural zeros, and no product formula can reproduce it. This notebook builds both tables and uses them to understand what independence looks like *inside* a joint distribution. It is based on the second video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

The last notebook built a joint distribution from data. This one builds two from pure theory — and they could not be more different:

- $X$ = first die, $Y$ = **second die**: knowing $X$ tells you nothing about $Y$.
- $X$ = first die, $Y$ = **sum of both**: knowing $X$ tells you a lot about $Y$ (the sum must be between $X+1$ and $X+6$).

After completing this notebook you will be able to:

- Write the joint PMF of two **independent** discrete variables as the product $P_{XY}(x,y) = P_X(x)P_Y(y)$.
- Build the joint distribution of the first die and the sum, including its structural zeros.
- Recognize dependence in a joint table at a glance — and test factorization cell by cell.

## 2. Theory

### Two dice: the independent case

Throw two fair six-sided dice. Let $X$ be the number rolled on the first die and $Y$ the number on the second. Each variable has the same PMF: probability $\tfrac16$ for each of $1, \dots, 6$.

$X$ and $Y$ are **independent** — so, for example,

$$
P(X = 2 \text{ and } Y = 5) = \frac{1}{6} \cdot \frac{1}{6} = \frac{1}{36}
$$

and the same happens for *every* pair: the joint PMF is simply a $6 \times 6$ table with $\tfrac{1}{36}$ everywhere. In formula form:

$$
P_{XY}(x, y) = P(X = x,\ Y = y) = P_X(x) \cdot P_Y(y) \qquad \text{(independent variables)}
$$

The joint PMF **factors** into the product of the individual PMFs. This will *not* be the case all the time — it is the defining special property of independence — but when it holds, joint distributions are as simple as they get.

### A more complicated example: the first die and the sum

Same two dice; keep $X$ = first die, but now let $Y$ = the **sum** of the two (first die 4, second die 5 → $Y = 9$).

The PMF of $Y$ alone comes from the full $6\times6$ table of outcomes: sums run from 2 (only $1{+}1$) to 12 (only $6{+}6$), and the histogram peaks at **7** — the sum with the most combinations — while 2 and 12 are the rarest.

Now the **joint** table of $(X, Y)$: put $X$ on one axis and the sum $Y$ on the other, and place each of the 36 outcomes at its coordinates — outcome $(1,1)$ sits at $x{=}1, y{=}2$; outcome $(1,2)$ at $x{=}1, y{=}3$; and so on. Two things appear:

- **Every occupied cell holds exactly one outcome** — only one combination of dice leads to each (first die, sum) pair — so each occupied cell has probability $\tfrac{1}{36}$.
- The occupied cells form a **diagonal band**: for a given $x$, only sums $x+1$ through $x+6$ are possible. Everything else is a **structural zero**.

### Reading the dependent table

- $P_{XY}(3, 7)$: third column, seventh row — an occupied cell → $\tfrac{1}{36}$. (Behind it: dice $3$ and $4$.)
- $P_{XY}(1, 1)$: first die 1 *and sum 1* is impossible — no die can roll a 0 — a zero cell → probability $0$.

And factorization fails: $P_X(1) \cdot P_Y(2) = \tfrac16 \cdot \tfrac1{36} = \tfrac1{216}$, but the joint value is $\tfrac{1}{36}$. Knowing $X = 1$ *changes* the distribution of the sum — that is dependence, printed right into the table.

## 3. Worked Examples

### Example 1: independent dice

$$
P_{XY}(2, 5) = P_X(2)\,P_Y(5) = \tfrac16 \cdot \tfrac16 = \tfrac{1}{36}
$$

— and identically for all 36 cells: a perfectly flat joint table.

### Example 2: the PMF of the sum

Counting combinations per sum: $2{:}1,\ 3{:}2,\ 4{:}3,\ 5{:}4,\ 6{:}5,\ 7{:}6,\ 8{:}5,\ 9{:}4,\ 10{:}3,\ 11{:}2,\ 12{:}1$. Dividing by 36: $P(Y{=}7) = \tfrac{6}{36} = \tfrac16$, the mode of the triangle.

### Example 3: lookups in the (first die, sum) table

$$
P_{XY}(3, 7) = \tfrac{1}{36} \qquad\qquad P_{XY}(1, 1) = 0
$$

The first is the single outcome (die 1 = 3, die 2 = 4); the second is impossible.

### Example 4: testing independence with one cell

One counterexample cell is enough to prove dependence: $P_{XY}(1, 2) = \tfrac{1}{36} \ne \tfrac{1}{216} = P_X(1) P_Y(2)$. (Beware the reverse: *some* cells can match by coincidence — e.g. $P_{XY}(1,7) = \tfrac1{36} = P_X(1)P_Y(7)$ exactly — so proving independence requires **all** cells to factor.)

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

faces = np.arange(1, 7)

### The independent joint: build it by the product rule

In [ ]:
pmf_die = pd.Series(1 / 6, index=faces)

# outer product of the two PMFs = joint PMF of independent variables
joint_indep = pd.DataFrame(np.outer(pmf_die, pmf_die), index=faces, columns=faces)
joint_indep.index.name = "X (die 1)"
joint_indep.columns.name = "Y (die 2)"

print("joint PMF of two independent dice (all cells 1/36 = 0.0278):")
print(joint_indep.round(4))
print(f"\nP(X=2, Y=5) = {joint_indep.loc[2, 5]:.4f}   (1/36 = {1/36:.4f})")
print(f"total = {joint_indep.values.sum():.4f}")

### The dependent joint: first die and the sum, by enumeration

In [ ]:
sums = np.arange(2, 13)
joint_sum = pd.DataFrame(0.0, index=sums, columns=faces)
joint_sum.index.name = "Y (sum)"
joint_sum.columns.name = "X (die 1)"

for d1 in faces:
    for d2 in faces:
        joint_sum.loc[d1 + d2, d1] += 1 / 36        # each outcome fills one cell

print("joint PMF of (first die, sum) -- a diagonal band of 1/36:")
print(joint_sum.round(3))
print(f"\nP(X=3, Y=7)  = {joint_sum.loc[7, 3]:.4f}   (1/36)")
print(f"P(X=1, Y=1)  = 0 by construction: sum 1 is not even a row -- impossible")
print(f"total = {joint_sum.values.sum():.4f}")

### The PMF of the sum, recovered from the table

Summing each row of the joint gives the distribution of $Y$ alone — the triangle peaking at 7.

In [ ]:
pmf_sum = joint_sum.sum(axis=1)
print("PMF of the sum (counts out of 36):")
for s in sums:
    print(f"  Y = {s:>2}: {pmf_sum[s]:.4f}  = {round(pmf_sum[s] * 36):>2}/36")

### Testing factorization cell by cell

For the independent table, every cell equals the product of its marginals; for the (die, sum) table, most cells fail — one failing cell already proves dependence.

In [ ]:
def factorization_report(joint, name):
    px = joint.sum(axis=0)                     # PMF of the column variable
    py = joint.sum(axis=1)                     # PMF of the row variable
    product = pd.DataFrame(np.outer(py, px), index=joint.index, columns=joint.columns)
    mismatches = int((~np.isclose(joint.values, product.values)).sum())
    print(f"{name}: {mismatches} of {joint.size} cells fail P(x,y) = P(x)P(y)"
          f" -> {'independent' if mismatches == 0 else 'DEPENDENT'}")

factorization_report(joint_indep, "two dice          ")
factorization_report(joint_sum, "first die and sum ")

print(f"\nexample cell: P(X=1, Y=2) = {joint_sum.loc[2, 1]:.4f} "
      f"but P(X=1)*P(Y=2) = {1/6 * 1/36:.4f}")
print(f"coincidence cell: P(X=1, Y=7) = {joint_sum.loc[7, 1]:.4f} "
      f"= P(X=1)*P(Y=7) = {1/6 * 6/36:.4f} -- single matches prove nothing!")

## 5. Visualizations

### The two joint tables side by side

Independence is visible: a flat carpet vs. a diagonal band full of zeros.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))

im0 = axes[0].imshow(joint_indep.values, cmap="Blues", vmin=0, vmax=1/36 * 1.4)
axes[0].set_title("X = die 1, Y = die 2:\nindependent -> flat table of 1/36")
axes[0].set_xticks(range(6)); axes[0].set_xticklabels(faces)
axes[0].set_yticks(range(6)); axes[0].set_yticklabels(faces)
axes[0].set_xlabel("Y (die 2)"); axes[0].set_ylabel("X (die 1)")

im1 = axes[1].imshow(joint_sum.values, cmap="Blues", vmin=0, vmax=1/36 * 1.4)
axes[1].set_title("X = die 1, Y = sum:\ndependent -> diagonal band + zeros")
axes[1].set_xticks(range(6)); axes[1].set_xticklabels(faces)
axes[1].set_yticks(range(len(sums))); axes[1].set_yticklabels(sums)
axes[1].set_xlabel("X (die 1)"); axes[1].set_ylabel("Y (sum)")

fig.colorbar(im1, ax=axes, label="probability", shrink=0.8)
plt.show()

### The sum's histogram with its dice pairs

The video's annotated histogram: each column stacked from the outcomes that produce that sum.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for s in sums:
    pairs = [(d1, s - d1) for d1 in faces if 1 <= s - d1 <= 6]
    for level, (d1, d2) in enumerate(pairs):
        ax.bar(s, 1, bottom=level, width=0.85, color="tab:blue",
               edgecolor="white", alpha=0.85)
        ax.text(s, level + 0.5, f"{d1},{d2}", ha="center", va="center",
                color="white", fontsize=9)
ax.set_xticks(sums)
ax.set_yticks(range(0, 7))
ax.set_xlabel("sum of the two dice")
ax.set_ylabel("number of combinations (out of 36)")
ax.set_title("The sum's distribution: 7 has the most combinations, 2 and 12 the fewest")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### Simulation: both joints emerge from the same 100,000 throws

In [ ]:
rng = np.random.default_rng(42)

d1 = rng.integers(1, 7, size=100_000)
d2 = rng.integers(1, 7, size=100_000)

emp_indep = pd.crosstab(d1, d2, normalize=True)
emp_sum = pd.crosstab(d1 + d2, d1, normalize=True)

print(f"independent table: max |empirical - 1/36| = "
      f"{np.abs(emp_indep.values - 1/36).max():.4f}")
print(f"(die, sum) table:  max |empirical - theory| = "
      f"{np.abs(emp_sum.reindex(index=sums, columns=faces, fill_value=0).values - joint_sum.values).max():.4f}")

## 6. Exercises

### Basic

**Exercise 1.** Using the product rule, write down $P_{XY}(6, 6)$ for two independent dice, and $P_{XY}(x, y)$ for a fair coin ($\tfrac12$ each side) thrown together with a fair die. How many cells does the coin-die joint table have, and what is in each?

**Exercise 2.** In the (first die, sum) table: what are $P_{XY}(6, 12)$, $P_{XY}(6, 5)$, and $P_{XY}(2, 8)$? For each, name the dice outcome(s) behind the cell or explain the zero.

### Intermediate

**Exercise 3.** For a given first die $x$, how many *nonzero* cells does column $x$ of the (die, sum) table contain, and which sums are they? Verify by code for all six columns. What rule generates the diagonal band?

**Exercise 4.** Build the joint PMF of $X$ = first die and $M$ = the **maximum** of the two dice. (Careful: unlike the sum, some cells now hold *more* than one outcome.) Verify the table sums to 1 and read off $P(X{=}3, M{=}3)$.

### Advanced

**Exercise 5.** Prove by exhaustive check that two dice are independent but (die, max) is not: run the factorization report on your Exercise-4 table. Then find one cell where the joint and the product *happen* to be closest, and one where they differ most.

**Exercise 6.** Three independent dice: the joint PMF of $(X_1, X_2, X_3)$ is a $6\times6\times6$ array with $\tfrac{1}{216}$ everywhere. Build it with an outer product, confirm it sums to 1, and use it to compute $P(X_1 {=} 1, X_2 {=} 2, X_3 {=} 3)$ and $P(\text{all three dice equal})$ — the product rule scales to any number of independent variables.

## 7. Solutions

In [ ]:
# Exercise 1: product-rule tables
print(f"Exercise 1: P(6, 6) = 1/6 * 1/6 = {1/6 * 1/6:.4f}")
coin = pd.Series(0.5, index=["H", "T"])
die = pd.Series(1 / 6, index=faces)
coin_die = pd.DataFrame(np.outer(coin, die), index=coin.index, columns=die.index)
print("coin-die joint (2 x 6 = 12 cells, each 1/2 * 1/6 = 1/12):")
print(coin_die.round(4))

In [ ]:
# Exercise 2: three lookups in the (die, sum) table
print(f"Exercise 2: P(6, 12) = {joint_sum.loc[12, 6]:.4f}  (dice 6 and 6)")
print(f"            P(6, 5)  = {joint_sum.loc[5, 6]:.4f}  (sum 5 needs die2 = -1: impossible)")
print(f"            P(2, 8)  = {joint_sum.loc[8, 2]:.4f}  (dice 2 and 6)")

In [ ]:
# Exercise 3: the band structure
print("Exercise 3:")
for x in faces:
    nonzero = joint_sum.index[joint_sum[x] > 0].tolist()
    print(f"  X = {x}: {len(nonzero)} nonzero cells, sums {nonzero}")
print("Rule: die 2 adds 1..6, so column x is nonzero exactly for sums x+1 .. x+6 --")
print("six cells per column, sliding down one row per column: the diagonal band.")

In [ ]:
# Exercise 4: first die and the maximum
maxes = faces
joint_max = pd.DataFrame(0.0, index=maxes, columns=faces)
joint_max.index.name = "M (max)"
joint_max.columns.name = "X (die 1)"
for a in faces:
    for b in faces:
        joint_max.loc[max(a, b), a] += 1 / 36

print("Exercise 4:")
print(joint_max.round(3))
print(f"total = {joint_max.values.sum():.3f}")
print(f"P(X=3, M=3) = {joint_max.loc[3, 3]:.4f} = 3/36: outcomes (3,1), (3,2), (3,3)")

In [ ]:
# Exercise 5: factorization on (die, max)
factorization_report(joint_max, "first die and max ")

px = joint_max.sum(axis=0)
pm = joint_max.sum(axis=1)
product = pd.DataFrame(np.outer(pm, px), index=maxes, columns=faces)
gap = (joint_max - product).abs()
closest = gap.stack().idxmin()
farthest = gap.stack().idxmax()
print(f"closest cell (M, X) = {closest}: gap {gap.stack().min():.4f}")
print(f"farthest cell (M, X) = {farthest}: gap {gap.stack().max():.4f}")

**Exercise 5, illustrated.** A one-page visual summary of the solution above.

<img src="img/02_ex5_factorization_die_and_max.png" alt="Illustrated solution to Exercise 5" width="900">

In [ ]:
# Exercise 6: three independent dice
joint3 = np.multiply.outer(np.outer(np.full(6, 1/6), np.full(6, 1/6)), np.full(6, 1/6))
print(f"Exercise 6: shape = {joint3.shape}, total = {joint3.sum():.4f}")
print(f"P(X1=1, X2=2, X3=3) = {joint3[0, 1, 2]:.6f}   (1/216 = {1/216:.6f})")
all_equal = sum(joint3[i, i, i] for i in range(6))
print(f"P(all three equal) = {all_equal:.4f}   (6/216 = {6/216:.4f})")
print("The product rule extends: for independent variables the joint is the outer")
print("product of the individual PMFs, in any dimension.")

## 8. Conclusion

- For **independent** discrete variables the joint PMF **factors**: $P_{XY}(x,y) = P_X(x)\,P_Y(y)$ — two fair dice give a flat $6\times6$ table of $\tfrac1{36}$.
- Factorization is special, not universal: with $Y$ = the **sum**, each (first die, sum) pair comes from exactly one outcome, giving a **diagonal band** of $\tfrac{1}{36}$-cells surrounded by structural zeros ($P_{XY}(3,7) = \tfrac1{36}$, $P_{XY}(1,1) = 0$).
- The sum's own PMF — the triangle peaking at 7 — is recovered by summing the joint's rows.
- Dependence shows up as factorization failure ($P_{XY}(1,2) = \tfrac1{36} \ne \tfrac1{216}$); but beware coincidental matches in single cells — independence requires **every** cell to factor.
- The product rule scales to any number of independent variables (three dice: $\tfrac1{216}$ per cell).

Next: **continuous** joint distributions — the same ideas with PDFs and volumes instead of tables.